# CoT Trace Audit — multi-run comparison

Compare teacher-generated traces across multiple runs on the same SURDS probe set.
Each run is a tuple of `(label, path, parse_mode)`.

- `think_answer` mode: fields `thinking`, `answer`, `leftover`.
- `structured` mode: fields `perception`, `grounding`, `inference`, `verification`, `answer`, `leftover`.


In [ ]:
import io
import json
import re
from pathlib import Path

import pandas as pd
from IPython.display import Image as IPyImage, Markdown, display
from PIL import Image as PILImage

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 200)

DATA = '/mnt/data4/shasta/amar.amarjyoti/research_data/vlm_cot_distill'

# Edit this list to compare different runs.
RUNS = [
    ('baseline',   f'{DATA}/outputs_1057764_Qwen3-VL-32B-Thinking_baseline-prompt_T0.6_N16384_reparsed.jsonl', 'think_answer'),
    ('tightened',  f'{DATA}/outputs_1057782_Qwen3-VL-32B-Thinking_tightened-prompt_T0.3_N6000.jsonl',          'think_answer'),
    ('structured', f'{DATA}/outputs_1057783_Qwen3-VL-32B-Thinking_structured-prompt_T0.3_N6000.jsonl',         'structured'),
    ('235B-unprompted', f'{DATA}/outputs_1057805_Qwen3-VL-235B-A22B-Thinking-FP8_no-system-prompt_T0.6_N16384_reparsed.jsonl', 'think_answer'),
]

In [ ]:
def load_run(path: str, parse_mode: str) -> list[dict]:
    rows = [json.loads(l) for l in Path(path).read_text().splitlines() if l.strip()]
    for r in rows:
        r['_parse_mode'] = parse_mode
    return rows

runs = {label: load_run(path, mode) for label, path, mode in RUNS}
by_tt = {label: {r['template_type']: r for r in rows} for label, rows in runs.items()}
template_order = ['distance', 'lr', 'fb', 'yaw', 'xy2d', 'depth']

for label, path, _ in RUNS:
    print(f'{label:12s}: {len(runs[label])} samples  ({Path(path).name})')

## Helpers

In [ ]:
def is_correct(r: dict) -> bool:
    return str(r.get('answer', '')).strip().lower() == str(r.get('gt_answer', '')).strip().lower()

def trace_text(r: dict) -> str:
    if r.get('_parse_mode') == 'structured':
        parts = []
        for sec in ['perception', 'grounding', 'inference', 'verification']:
            val = r.get(sec, '').strip()
            if val:
                parts.append(f'### `<{sec}>`\n\n{val}')
        return '\n\n'.join(parts) if parts else '_[no structured sections parsed — see raw]_'
    return r.get('thinking', '') or '_[no <think> content]_'

def trace_char_count(r: dict) -> int:
    if r.get('_parse_mode') == 'structured':
        return sum(len(r.get(s, '')) for s in ['perception', 'grounding', 'inference', 'verification'])
    return len(r.get('thinking', ''))

def render_image(path: str, width: int = 720):
    try:
        pil = PILImage.open(path).convert('RGB')
        pil.thumbnail((width, width))
        buf = io.BytesIO()
        pil.save(buf, format='PNG')
        display(IPyImage(data=buf.getvalue()))
    except Exception as e:
        print(f'[image load failed: {e}]  path={path}')

## 1. Aggregate metrics per run

In [ ]:
agg_rows = []
for label, rows in runs.items():
    n = len(rows)
    n_correct = sum(is_correct(r) for r in rows)
    n_answer_present = sum(1 for r in rows if r.get('answer', '').strip())
    out_toks = [r.get('num_output_tokens', 0) for r in rows]
    trace_chars = [trace_char_count(r) for r in rows]
    agg_rows.append({
        'run': label,
        'n': n,
        'correct': f'{n_correct}/{n} ({n_correct/n:.0%})',
        'answer_present': f'{n_answer_present}/{n}',
        'out_toks_mean': int(sum(out_toks) / n),
        'out_toks_max': max(out_toks),
        'trace_chars_mean': int(sum(trace_chars) / n),
        'trace_chars_max': max(trace_chars),
    })
pd.DataFrame(agg_rows)

## 2. Per-sample answer table
Quick scan of which runs got which samples right.

In [ ]:
rows_out = []
for tt in template_order:
    first_run = list(by_tt)[0]
    gt = by_tt[first_run][tt]['gt_answer']
    row = {'template_type': tt, 'gt': gt}
    for label in by_tt:
        r = by_tt[label].get(tt)
        if r is None:
            row[label] = '—'
            continue
        ok = '✓' if is_correct(r) else '✗'
        ans = r.get('answer', '')[:40]
        row[label] = f"{ok} {ans}  ({trace_char_count(r)}ch / {r.get('num_output_tokens',0)}t)"
    rows_out.append(row)
pd.DataFrame(rows_out)

## 3. Per-sample deep-dive
For each probe: image once, prompt, then each run's answer and trace.
Use this to understand *why* one prompt wins or loses on a given task type.

In [ ]:
def show_compare(tt: str):
    first_run = list(by_tt)[0]
    r0 = by_tt[first_run][tt]
    gt = r0['gt_answer']
    q = re.sub(r'Reason carefully.*', '', r0['prompt'], flags=re.DOTALL).strip()
    display(Markdown(f'# `{tt}` — GT: `{gt}`'))
    render_image(r0['image_path'])
    display(Markdown(f"**Prompt:**\n\n```\n{q}\n```"))
    for label in by_tt:
        r = by_tt[label].get(tt)
        if r is None:
            continue
        ok = '✅' if is_correct(r) else '❌'
        header = (
            f"## {ok} {label}\n"
            f"- **answer:** `{r.get('answer', '')}`  (GT: `{gt}`)\n"
            f"- **trace chars:** {trace_char_count(r):,}  |  **output tokens:** {r.get('num_output_tokens', 0)}  |  **finish:** `{r.get('finish_reason','')}`\n"
        )
        display(Markdown(header))
        display(Markdown(trace_text(r)))
        if r.get('leftover'):
            display(Markdown(f"_Leftover outside tags:_ `{r['leftover'][:400]}`"))
    display(Markdown('---'))

for tt in template_order:
    show_compare(tt)

## 4. Format failure drill-down
Any run / sample with empty answer, truncation, stray content, or missing structured sections.

In [ ]:
issues = []
for tt in template_order:
    for label in by_tt:
        r = by_tt[label].get(tt)
        if r is None:
            continue
        problems = []
        if not r.get('answer', '').strip():
            problems.append('empty_answer')
        if r.get('leftover') and len(r['leftover']) > 50:
            problems.append(f"leftover={len(r['leftover'])}ch")
        if r.get('finish_reason') == 'length':
            problems.append('truncated')
        if r.get('_parse_mode') == 'structured':
            missing = [s for s in ['perception','grounding','inference','verification'] if not r.get(s,'').strip()]
            if missing:
                problems.append(f'missing={missing}')
        if problems:
            issues.append({'template_type': tt, 'run': label, 'answer': r.get('answer',''), 'issues': problems})
pd.DataFrame(issues) if issues else 'No format issues found.'

## 5. Raw output dump
Pick any `(run_label, template_type)` to inspect the untouched model output.

In [ ]:
RUN_TO_INSPECT = 'structured'
TT_TO_INSPECT = 'xy2d'

r = by_tt[RUN_TO_INSPECT][TT_TO_INSPECT]
print(r['raw'])

## 6. val3k baseline analysis

<details>
<summary><b>Summary table — 3000 SURDS validation samples (500 / template_type), 4 runs</b></summary>

| template | 32B-referent | 235B-referent | 32B-grounding | 235B-grounding |
|----------|:---:|:---:|:---:|:---:|
| lr       | 86.8% | **88.6%** | 84.2% | 88.4% |
| distance | **80.6%** | 78.0% | 61.6% | 79.0% |
| fb       | 34.6% | 49.0% | 26.6% | **52.4%** |
| depth    | **48.2%** | 45.8% | 36.8% | 43.4% |
| yaw      | 38.0% | **43.0%** | 34.6% | 40.0% |
| xy2d     | 1.2% | 5.6% | 1.4% | **6.0%** |
| **overall** | 48.2% | **51.7%** | 40.9% | 51.5% |

(xy2d uses 50-pixel L∞ tolerance; exact-match is ≈0% everywhere.)

**Takeaways**
- 235B beats 32B by ~3.5 pts overall; gains concentrated on fb (+15), yaw (+5), xy2d (+4).
- Grounding prompt **breaks 32B format compliance** (207 empty answers, -7.3 pts). Roughly neutral on 235B.
- xy2d is broken everywhere — a separate failure mode worth investigating.
- 32B-referent is a strong cheap baseline; 235B-referent is the best teacher overall.

</details>


### 6.1 Load val3k runs

In [ ]:
from collections import Counter, defaultdict
import re

VAL3K = {
    '32B-ref':    f'{DATA}/outputs_1057887_Qwen3-VL-32B-Thinking_val3k_referent-tolerant_T0.6_N16384.jsonl',
    '235B-ref':   f'{DATA}/outputs_1057888_Qwen3-VL-235B-A22B-Thinking-FP8_val3k_referent-tolerant_T0.6_N16384.jsonl',
    '32B-ground': f'{DATA}/outputs_1058076_Qwen3-VL-32B-Thinking_grounding-prompt_T0.6_N16384_val3k.jsonl',
    '235B-ground':f'{DATA}/outputs_1058077_Qwen3-VL-235B-A22B-Thinking-FP8_grounding-prompt_T0.6_N16384_val3k.jsonl',
}

ANSWER_RE = re.compile(r'<answer>\s*(.*?)\s*</answer>', re.S)

def get_ans(r):
    a = (r.get('answer') or '').strip()
    if a:
        return a
    raw = r.get('raw') or r.get('output') or ''
    m = ANSWER_RE.search(raw)
    return (m.group(1).strip() if m else '')

def norm(s):
    return re.sub(r'\s+', ' ', (s or '').strip().lower())

val3k = {name: [json.loads(l) for l in Path(p).read_text().splitlines() if l.strip()]
         for name, p in VAL3K.items()}
for name, rows in val3k.items():
    for r in rows:
        r['_ans'] = get_ans(r)
print({k: len(v) for k, v in val3k.items()})


### 6.2 Per-template accuracy (lenient)

In [ ]:
from PIL import Image as PILImageMod

def _img_dims(path, _cache={}):
    if path not in _cache:
        try: _cache[path] = PILImageMod.open(path).size
        except Exception: _cache[path] = (1600, 900)
    return _cache[path]

def _parse_xy_int(s):
    nums = re.findall(r'-?\d+', s or '')
    if len(nums) >= 2:
        try: return int(nums[0]), int(nums[1])
        except: return None
    return None

def correct_lenient(pred, gt, tt, image_path=None, xy_tol=50, rescale_xy_from_1000=True):
    """Lenient correctness. For xy2d, rescale predicted [0,1000] coords to image pixels
    using the actual image dimensions before applying tolerance."""
    p, g = norm(pred), norm(str(gt))
    if not p:
        return False
    if tt == 'xy2d':
        pp = _parse_xy_int(p); gg = _parse_xy_int(g)
        if pp is None or gg is None:
            return False
        if rescale_xy_from_1000 and image_path is not None:
            W, H = _img_dims(image_path)
            pp = (pp[0] * W / 1000.0, pp[1] * H / 1000.0)
        return abs(pp[0] - gg[0]) <= xy_tol and abs(pp[1] - gg[1]) <= xy_tol
    return p == g or g in p

acc_table = []
for name, rows in val3k.items():
    by_tt = defaultdict(lambda: [0, 0])
    for r in rows:
        tt = r.get('template_type', '?')
        by_tt[tt][1] += 1
        if correct_lenient(r['_ans'], r.get('gt_answer',''), tt, image_path=r.get('image_path')):
            by_tt[tt][0] += 1
    row = {'run': name}
    tot_c = tot_n = 0
    for tt in sorted(by_tt):
        c, n = by_tt[tt]
        row[tt] = f'{c/n*100:.1f}%'
        tot_c += c; tot_n += n
    row['overall'] = f'{tot_c/tot_n*100:.1f}%'
    acc_table.append(row)
pd.DataFrame(acc_table).set_index('run')

### 6.3 xy2d failure mode — pixel error distribution

xy2d asks for `[x, y]` integers. Lenient match was 50px L∞; ground truth points are sub-pixel-precise. Look at the *distance* between predicted and GT points to see whether the model is roughly right or completely lost.

In [ ]:
def parse_xy(s):
    nums = re.findall(r'-?\d+', s or '')
    if len(nums) >= 2:
        try: return int(nums[0]), int(nums[1])
        except: return None
    return None

xy_stats = []
for name, rows in val3k.items():
    errs = []
    parse_fail = 0
    for r in rows:
        if r.get('template_type') != 'xy2d': continue
        p = parse_xy(r['_ans']); g = parse_xy(str(r.get('gt_answer','')))
        if p is None or g is None:
            parse_fail += 1; continue
        errs.append(max(abs(p[0]-g[0]), abs(p[1]-g[1])))
    s = pd.Series(errs)
    xy_stats.append({
        'run': name,
        'n_with_pred': len(errs),
        'parse_fail': parse_fail,
        'median_err_px': int(s.median()) if len(s) else None,
        'p25': int(s.quantile(0.25)) if len(s) else None,
        'p75': int(s.quantile(0.75)) if len(s) else None,
        'p95': int(s.quantile(0.95)) if len(s) else None,
        'pct_under_50px': f'{(s<=50).mean()*100:.1f}%' if len(s) else None,
        'pct_under_100px': f'{(s<=100).mean()*100:.1f}%' if len(s) else None,
        'pct_under_200px': f'{(s<=200).mean()*100:.1f}%' if len(s) else None,
    })
pd.DataFrame(xy_stats).set_index('run')


### 6.4 yaw failure mode — confusion matrix
GT direction (rows) vs predicted direction (cols) for the best run.

In [ ]:
COMPASS = ['North','Northeast','East','Southeast','South','Southwest','West','Northwest']

def canon_compass(s):
    s = norm(s)
    for d in COMPASS:
        if d.lower() == s: return d
    for d in COMPASS:
        if d.lower() in s: return d
    return 'OTHER/empty'

def yaw_confusion(run_name):
    rows = [r for r in val3k[run_name] if r.get('template_type')=='yaw']
    cm = pd.DataFrame(0, index=COMPASS+['OTHER/empty'], columns=COMPASS+['OTHER/empty'])
    for r in rows:
        gt = canon_compass(str(r.get('gt_answer','')))
        pr = canon_compass(r['_ans'])
        cm.loc[gt, pr] += 1
    return cm

display(Markdown('**235B-ref yaw confusion (rows=GT, cols=pred)**'))
yaw_confusion('235B-ref')


In [ ]:
display(Markdown('**32B-ref yaw confusion**'))
yaw_confusion('32B-ref')


### 6.5 fb failure mode — Yes/No/Almost-same confusion

fb has only 3 GT classes. Where does it fail?

In [ ]:
def canon_fb(s):
    s = norm(s)
    if 'almost' in s: return 'Almost the same'
    if s.startswith('yes'): return 'Yes'
    if s.startswith('no'): return 'No'
    if not s: return 'EMPTY'
    return 'OTHER'

def fb_confusion(run_name):
    rows = [r for r in val3k[run_name] if r.get('template_type')=='fb']
    cats = ['Yes','No','Almost the same','OTHER','EMPTY']
    cm = pd.DataFrame(0, index=cats, columns=cats)
    for r in rows:
        gt = canon_fb(str(r.get('gt_answer','')))
        pr = canon_fb(r['_ans'])
        if gt not in cats: gt = 'OTHER'
        cm.loc[gt, pr] += 1
    return cm

for run in ['32B-ref','235B-ref','32B-ground','235B-ground']:
    display(Markdown(f'**{run} fb confusion**'))
    display(fb_confusion(run))


### 6.6 depth failure mode — bucket overlap

depth GT is `Between X meters and Y meters`. Check whether the predicted bucket overlaps GT, is one bucket off, or completely wrong.

In [ ]:
BUCKET_RE = re.compile(r'between\s+(\d+)\s*meters?\s+and\s+(\d+)\s*meters?', re.I)

def parse_bucket(s):
    m = BUCKET_RE.search(s or '')
    if not m: return None
    return int(m.group(1)), int(m.group(2))

def depth_breakdown(run_name):
    rows = [r for r in val3k[run_name] if r.get('template_type')=='depth']
    counts = Counter()
    for r in rows:
        g = parse_bucket(str(r.get('gt_answer',''))); p = parse_bucket(r['_ans'])
        if p is None:
            counts['unparseable'] += 1; continue
        if g is None:
            counts['gt_unparseable'] += 1; continue
        if p == g:
            counts['exact'] += 1
        elif max(p[0], g[0]) <= min(p[1], g[1]):
            counts['overlapping'] += 1
        else:
            gap = min(abs(p[0]-g[1]), abs(g[0]-p[1]))
            if gap <= 5: counts['off_<=5m'] += 1
            elif gap <= 15: counts['off_6_15m'] += 1
            else: counts['off_>15m'] += 1
    return counts

depth_df = pd.DataFrame({run: depth_breakdown(run) for run in val3k}).fillna(0).astype(int).T
depth_df['n'] = depth_df.sum(axis=1)
depth_df


### 6.7 lr / distance failure mode — categorical referent picks

lr and distance ask the model to pick which described object is leftmost / closest. Failures usually mean the model picked a different referent (or hallucinated one). Tabulate the most common (GT, pred) confusion pairs.

In [ ]:
def categorical_confusions(run_name, tt, top_n=15):
    rows = [r for r in val3k[run_name] if r.get('template_type')==tt]
    pairs = Counter()
    for r in rows:
        gt = norm(str(r.get('gt_answer','')))
        pr = norm(r['_ans'])
        if not pr:
            pairs[(gt, '<EMPTY>')] += 1
        elif gt != pr and gt not in pr:
            pairs[(gt, pr[:60])] += 1
    return pd.DataFrame([{'gt': g, 'pred': p, 'n': c} for (g,p),c in pairs.most_common(top_n)])

display(Markdown('**235B-ref — top lr confusions**'))
display(categorical_confusions('235B-ref', 'lr'))
display(Markdown('**235B-ref — top distance confusions**'))
display(categorical_confusions('235B-ref', 'distance'))


### 6.8 Sample failure traces

Pull a handful of wrong-answer traces per template from the best run, to inspect the reasoning.

In [ ]:
def sample_failures(run_name, tt, k=3):
    rows = [r for r in val3k[run_name] if r.get('template_type')==tt
            and not correct_lenient(r['_ans'], r.get('gt_answer',''), tt)]
    return rows[:k]

INSPECT_RUN = '235B-ref'
INSPECT_TT = 'xy2d'

for r in sample_failures(INSPECT_RUN, INSPECT_TT, k=3):
    display(Markdown(f"**id={r.get('id','?')}  GT=`{r.get('gt_answer')}`  pred=`{r['_ans']}`**"))
    render_image(r['image_path'])
    th = (r.get('thinking') or '').strip()
    print((th[:1200] + ' …[truncated]') if len(th) > 1200 else th)
    display(Markdown('---'))


### 6.9 xy2d coordinate-convention investigation

<details>
<summary><b>Findings</b></summary>

The 1–6% lenient xy2d accuracy was a **units bug**, not a perception failure.

- Predictions are bounded: `pred_x ∈ [0, 985]`, `pred_y ∈ [0, 860]` — Qwen-VL's inherited **[0, 1000] normalized coordinate convention**.
- GT is in **raw pixel coordinates** of the original 1600×900 nuScenes camera frame (`gt_x` up to 1597, `gt_y` up to 893).
- Smart-resize is **not** the cause: Qwen3-VL processor reports `image_grid_thw=[1, 56, 100]` (patch=16, merge=2) ⇒ resized image is 1600×896, essentially native. So "resized pixels" and "raw pixels" give identical errors.

After rescaling `x_orig = pred_x * W/1000, y_orig = pred_y * H/1000`:

| run | raw med err | rescaled med err | <=50 px raw | <=50 px rescaled | <=100 px rescaled |
|---|---|---|---|---|---|
| 32B-ref      | 331 | 129 | 1.2% | 14.8% | 37.9% |
| **235B-ref**     | 316 | **15** | 5.7% | **76.7%** | **84.2%** |
| 32B-ground   | 330 | 151 | 1.4% | 8.9% | 28.4% |
| **235B-ground**  | 316 | **15** | 6.2% | **78.9%** | **87.1%** |

**Implications**
- 235B is an excellent xy2d teacher once descaled (median 15 px on 1600×900) — distillation-grade.
- 32B is convention-inconsistent (median 129–151 px after rescale) — mixes normalized and raw on different samples.
- For CoT distillation: post-process the `<answer>` field with `(W/1000, H/1000)` rescale before saving as student targets, OR pass image dimensions into the prompt and require raw pixels.

</details>

In [ ]:
from PIL import Image as PILImageMod

def img_dims(path, _cache={}):
    if path not in _cache:
        try: _cache[path] = PILImageMod.open(path).size
        except Exception: _cache[path] = (1600, 900)
    return _cache[path]

def parse_xy(s):
    nums = re.findall(r'-?\d+', s or '')
    if len(nums) >= 2:
        try: return int(nums[0]), int(nums[1])
        except: return None
    return None

def xy2d_rescaled_stats(rows, scale_to_xy1000=True):
    raw_errs, resc_errs = [], []
    for r in rows:
        if r.get('template_type') != 'xy2d': continue
        p = parse_xy(r['_ans']); g = parse_xy(str(r.get('gt_answer','')))
        if not (p and g): continue
        W, H = img_dims(r['image_path'])
        raw_errs.append(max(abs(p[0]-g[0]), abs(p[1]-g[1])))
        px, py = (p[0]*W/1000.0, p[1]*H/1000.0) if scale_to_xy1000 else p
        resc_errs.append(max(abs(px-g[0]), abs(py-g[1])))
    s_raw = pd.Series(raw_errs); s_rsc = pd.Series(resc_errs)
    return {
        'n': len(raw_errs),
        'raw_med': int(s_raw.median()),
        'raw_<=50': f'{(s_raw<=50).mean()*100:.1f}%',
        'raw_<=100': f'{(s_raw<=100).mean()*100:.1f}%',
        'rescaled_med': int(s_rsc.median()),
        'rescaled_<=50': f'{(s_rsc<=50).mean()*100:.1f}%',
        'rescaled_<=100': f'{(s_rsc<=100).mean()*100:.1f}%',
    }

xy_resc = pd.DataFrame({name: xy2d_rescaled_stats(rows) for name, rows in val3k.items()}).T
xy_resc

**Per-sample bimodality check (32B):** does it sometimes emit raw pixels and sometimes normalized? Plot rescaled-error histogram.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(18, 3.5), sharey=True)
for ax, (name, rows) in zip(axes, val3k.items()):
    errs = []
    for r in rows:
        if r.get('template_type') != 'xy2d': continue
        p = parse_xy(r['_ans']); g = parse_xy(str(r.get('gt_answer','')))
        if not (p and g): continue
        W, H = img_dims(r['image_path'])
        errs.append(min(max(abs(p[0]*W/1000-g[0]), abs(p[1]*H/1000-g[1])), 1500))
    ax.hist(errs, bins=40, range=(0, 1500))
    ax.axvline(50, color='g', ls='--', alpha=0.5)
    ax.axvline(100, color='orange', ls='--', alpha=0.5)
    ax.set_title(f'{name}\nn={len(errs)}')
    ax.set_xlabel('rescaled L∞ pixel error')
axes[0].set_ylabel('count')
plt.tight_layout()
plt.show()

## 7. Grounding-prompt deep-dive (6 examples, one per template_type)

Same shape as section 3 but using the **grounding-prompt** val3k runs (32B vs 235B).
Picks one representative sample per template_type from val3k for side-by-side inspection of the trace + answer.

In [ ]:
GROUND_RUNS = {
    '32B-ground':  val3k['32B-ground'],
    '235B-ground': val3k['235B-ground'],
}

# Pick one sample per template_type — prefer one where 235B is correct and 32B is wrong (most informative)
ground_probes = {}
for tt in ['distance', 'lr', 'fb', 'yaw', 'xy2d', 'depth']:
    by_id_32 = {r['id']: r for r in GROUND_RUNS['32B-ground'] if r.get('template_type')==tt}
    by_id_235 = {r['id']: r for r in GROUND_RUNS['235B-ground'] if r.get('template_type')==tt}
    common = set(by_id_32) & set(by_id_235)
    pick = None
    for sid in common:
        ok32 = correct_lenient(by_id_32[sid]['_ans'], by_id_32[sid].get('gt_answer',''), tt)
        ok235 = correct_lenient(by_id_235[sid]['_ans'], by_id_235[sid].get('gt_answer',''), tt)
        if ok235 and not ok32:
            pick = sid; break
    if pick is None and common:
        pick = sorted(common)[0]
    if pick:
        ground_probes[tt] = pick
print(ground_probes)

In [ ]:
GROUNDING_RE = re.compile(r'<grounding>(.*?)</grounding>', re.S)
THINK_RE2   = re.compile(r'<think>(.*?)</think>', re.S)
ANSWER_RE2  = re.compile(r'<answer>(.*?)</answer>', re.S)

def split_grounding_trace(raw: str):
    """Return (grounding, thinking, answer) from a grounding-prompt raw output."""
    g = GROUNDING_RE.search(raw or '')
    grounding = g.group(1).strip() if g else ''
    # Strip grounding block before extracting think
    rest = GROUNDING_RE.sub('', raw or '')
    # Qwen3-Thinking quirk: only </think> is in output; everything before it is thinking
    close = rest.find('</think>')
    if close >= 0:
        open_ = rest.find('<think>')
        thinking = rest[open_+len('<think>'):close].strip() if 0 <= open_ < close else rest[:close].strip()
        tail = rest[close+len('</think>'):]
    else:
        m = THINK_RE2.search(rest)
        thinking = m.group(1).strip() if m else ''
        tail = THINK_RE2.sub('', rest)
    a = ANSWER_RE2.search(tail)
    answer = a.group(1).strip() if a else ''
    return grounding, thinking, answer

def show_ground_compare(tt: str):
    sid = ground_probes.get(tt)
    if not sid:
        display(Markdown(f'## `{tt}` — no overlapping sample found')); return
    r32 = next(r for r in GROUND_RUNS['32B-ground'] if r['id']==sid)
    r235 = next(r for r in GROUND_RUNS['235B-ground'] if r['id']==sid)
    gt = r32.get('gt_answer')
    q = re.sub(r'Reason carefully.*', '', r32.get('prompt',''), flags=re.DOTALL).strip()
    display(Markdown(f'# `{tt}` — id `{sid}`  GT: `{gt}`'))
    render_image(r32['image_path'])
    display(Markdown(f"**Question**\n\n{q}"))
    for label, r in [('32B-ground', r32), ('235B-ground', r235)]:
        ok = '✓' if correct_lenient(r['_ans'], gt, tt) else '✗'
        grounding, thinking, answer = split_grounding_trace(r.get('raw',''))
        display(Markdown(f"### {label} — answer: `{r['_ans']}`  {ok}"))
        display(Markdown(f"**<grounding>** ({len(grounding)} chars)"))
        print(grounding[:2000] + (' …[truncated]' if len(grounding)>2000 else '') if grounding else '<empty>')
        display(Markdown(f"**<think>** ({len(thinking)} chars)"))
        print(thinking[:2000] + (' …[truncated]' if len(thinking)>2000 else '') if thinking else '<empty>')
        display(Markdown('---'))

for tt in ['distance', 'lr', 'fb', 'yaw', 'xy2d', 'depth']:
    show_ground_compare(tt)